# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

I chose a **ranking model** approach because the practical decision is which content pages should be reviewed first. I will create a transparent score from historical, pre-decision signals and rank pages by their estimated risk of a meaningful future impression decline.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

My target is a binary future-decline label. A page is labeled 1 when its impressions in the target month are less than 80% of its impressions in the preceding feature month, provided it had at least 100 impressions in the feature month. The target is calculated from the later target window in the warehouse `fact_content_daily_performance` table, while all model features come from the earlier window.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

My primary success metric is **precision@20**. A good result means that a useful share of the 20 highest-ranked pages actually experience a meaningful impression decline in the following month. I will compare the model with the base rate and a simple baseline. A result is useful only if the top 20 pages contain more future declines than would be expected from selecting 20 eligible pages at random.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of analysis:** One row represents one content item for one client, identified by `client_hash_id` and `content_hash_id`, with its daily records aggregated over the feature window.

The features are calculated only from the earlier window, such as **February 2026**. The target is calculated separately from the later target window, such as **March 2026**. The target is future_decline_label, equal to 1 when March impressions are less than 80% of February impressions, for pages with at least 100 February impressions.

The IDs are used for grouping and client-based splitting only. They are not model features.

In [5]:
%pip -q install duckdb huggingface_hub python-dotenv pyarrow

import os
import duckdb
from dotenv import load_dotenv

load_dotenv()
HF_TOKEN = os.environ.get('HF_TOKEN')

con = duckdb.connect()
if HF_TOKEN:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

february_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet"
march_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

warehouse_query = f"""
WITH february AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS prior_impressions,
        SUM(gsc_clicks) AS prior_clicks,
        SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS prior_avg_position,
        SUM(ga4_sessions) AS prior_sessions,
        SUM(ga4_engaged_sessions) / NULLIF(SUM(ga4_sessions), 0) AS prior_engagement_rate
    FROM read_parquet('{february_path}')
    WHERE report_date >= DATE '2026-02-01'
      AND report_date < DATE '2026-03-01'
    GROUP BY client_hash_id, content_hash_id
),
march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS future_impressions
    FROM read_parquet('{march_path}')
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    february.client_hash_id,
    february.content_hash_id,
    february.prior_impressions,
    february.prior_clicks,
    february.prior_avg_position,
    february.prior_sessions,
    february.prior_engagement_rate,
    COALESCE(march.future_impressions, 0) AS future_impressions,
    CASE
        WHEN february.prior_impressions >= 100
         AND COALESCE(march.future_impressions, 0) < 0.8 * february.prior_impressions
        THEN 1
        WHEN february.prior_impressions >= 100
        THEN 0
        ELSE NULL
    END AS future_decline_label
FROM february
LEFT JOIN march
    ON february.client_hash_id = march.client_hash_id
   AND february.content_hash_id = march.content_hash_id
"""

dataframe = con.execute(warehouse_query).df()

print(f"Rows in the February-to-March sample: {len(dataframe):,}")
print(f"Eligible rows with at least 100 prior impressions: {dataframe['future_decline_label'].notna().sum():,}")
print(f"Future decline rate among eligible rows: {dataframe['future_decline_label'].mean():.1%}")
dataframe.head()

Note: you may need to restart the kernel to use updated packages.
Rows in the February-to-March sample: 321,546
Eligible rows with at least 100 prior impressions: 80,322
Future decline rate among eligible rows: 21.8%


,client_hash_id,content_hash_id,prior_impressions,prior_clicks,prior_avg_position,prior_sessions,prior_engagement_rate,future_impressions,future_decline_label
0,client_3ffa76342f366962,content_3c3439c4de063402,0.0,0.0,NaN,NaN,NaN,0.0,<NA>
1,client_3ffa76342f366962,content_5e8730944999ba50,0.0,0.0,NaN,NaN,NaN,0.0,<NA>
2,client_3ffa76342f366962,content_427a1eabfe8a98f0,0.0,0.0,NaN,NaN,NaN,0.0,<NA>
3,client_3ffa76342f366962,content_bc411be5636aa6e8,0.0,0.0,NaN,NaN,NaN,0.0,<NA>
4,client_3ffa76342f366962,content_d745b8d2ef9b0419,6.0,0.0,16.333333,0.0,NaN,2.0,<NA>


In [7]:
from pathlib import Path

# Save the prepared February-to-March dataset once.
# Later notebooks can read this local file without querying Hugging Face again.
repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / 'work' / 'notebooks').exists() and (candidate / 'data').exists():
        repo_root = candidate
        break

output_path = repo_root / 'work' / 'outputs' / 'february_march_features.parquet'
output_path.parent.mkdir(parents=True, exist_ok=True)
dataframe.to_parquet(output_path, index=False)

print(f"Saved prepared dataset to: {output_path}")
print(f"Saved rows: {len(dataframe):,}")
print(f"Saved columns: {len(dataframe.columns)}")

Saved prepared dataset to: C:\Users\DELL\Documents\flyrank-ml-internship-starter\work\outputs\february_march_features.parquet
Saved rows: 321,546
Saved columns: 9


In [2]:
# Show the final row structure used for the Week 2 framing.
# IDs identify the page and client for grouping and later client-based splitting.
# They are not model features.
feature_columns = [
    'prior_impressions',
    'prior_clicks',
    'prior_avg_position',
    'prior_sessions',
    'prior_engagement_rate',
]

dataframe[feature_columns + ['future_impressions', 'future_decline_label']].head()

,prior_impressions,prior_clicks,prior_avg_position,prior_sessions,prior_engagement_rate,future_impressions,future_decline_label
0,0.0,0.0,NaN,NaN,NaN,0.0,<NA>
1,0.0,0.0,NaN,NaN,NaN,0.0,<NA>
2,0.0,0.0,NaN,NaN,NaN,0.0,<NA>
3,0.0,0.0,NaN,NaN,NaN,0.0,<NA>
4,6.0,0.0,16.333333,0.0,NaN,2.0,<NA>


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule may miss combinations of signals that indicate risk and may be too rigid in its decision processes. On the other hand, a transparent ranking model can combine several pre-decision signals and produce an ordered review list. I will only claim that it improves prioritization if it beats the simple baseline on the same time-based test set.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.